## 0. Import

In [1]:
# # DOWNLOAD DATASET 
# import kagglehub

# # Thay đường dẫn thư mục ở đây
# target_dir = "/home/pc/AAA/project/deepnn/forecast-practice/dataset"

# path = kagglehub.dataset_download(
#     "youssef019/mnist-multi-digit-regression-dataset",
#     output_dir=target_dir
# )

# print("Path to dataset files:", path)

In [2]:
import numpy as np
import pandas as pd
from PIL import Image
from math import sqrt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Permute, Reshape, Dense,
                                     Embedding, LSTM, Attention, Concatenate,
                                     GlobalAveragePooling1D, BatchNormalization, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

I0000 00:00:1795193273.239189   67244 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795193273.265657   67244 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795193273.826052   67244 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
# TensorFlow tự động dùng GPU khi phát hiện GPU tương thích.
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if gpus:
    print(f'Đã phát hiện {len(gpus)} GPU: {[gpu.name for gpu in gpus]}')
else:
    print('Không phát hiện GPU; TensorFlow sẽ chạy bằng CPU.')

Không phát hiện GPU; TensorFlow sẽ chạy bằng CPU.


W0000 00:00:1795193274.220783   67244 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
IMG_H, IMG_W = 28, 224
PAD_TOKEN = 10
START_TOKEN = 11                  # token bắt đầu cho decoder
N_CLASSES = 11              # 0-9 và PAD

## 1. Read dataset

In [5]:
df = pd.read_csv("/home/pc/AAA/project/deepnn/forecast-practice/dataset/labels.csv", dtype=str) # đọc dạng chuỗi để không mất số 0 ở đầu nhãn
print(df.head(), end="\n\n")
print(df.info())

   image     label
0  0.png   6290448
1  1.png    101651
2  2.png  36731246
3  3.png  24680373
4  4.png   8868017

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   image   60000 non-null  str  
 1   label   60000 non-null  str  
dtypes: str(2)
memory usage: 937.6 KB
None


In [6]:
images = []
labels = []

for image in df['image']:
    # thay đường dân folder ảnh vào đây
    image_path = "/home/pc/AAA/project/deepnn/forecast-practice/dataset/images/" + image 

    with Image.open(image_path) as img:
        img = img.convert('L')
        img = img.resize((IMG_W, IMG_H))
        img_arr = np.array(img, dtype=np.float32) / 255 # chuyển ảnh thành ma trận số và scale về [0, 1]
        images.append(img_arr)
images = np.array(images, dtype=np.float32)

for label in df['label']:
    labels.append([int(digit) for digit in label])
    
max_len = max(len(label) for label in labels)

print("Number of images:", len(images))
print("Image shape:", images.shape)
print("Maximum label length:", max_len)

Number of images: 60000
Image shape: (60000, 28, 224)
Maximum label length: 8


## 2. Preprocessing

In [7]:
# Thêm chiều kênh màu (N, 64, 128, 1) cho mạng CNN
images = np.expand_dims(np.array(images), axis=-1)

In [8]:
# Padding nhãn cho Encoder
y = np.full((len(labels), max_len), PAD_TOKEN, dtype='int32')
for i, label in enumerate(labels):
    y[i, :len(label)] = label

In [9]:
# Tạo đầu vào cho Decoder:
# Dịch nhãn sang phải 1 bước và chèn START_TOKEN vào đầu
start_col = np.full((len(y), 1), START_TOKEN, dtype='int32')
dec_in = np.concatenate([start_col, y[:, :-1]], axis=1)

## 3. Data Split

Chia train/val/test = 70/15/15

In [10]:
idx = np.arange(len(images))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=42)

X_train, X_val, X_test = images[idx_train], images[idx_val], images[idx_test]
D_train, D_val, D_test = dec_in[idx_train], dec_in[idx_val], dec_in[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

Train: (42000, 28, 224, 1)
Validation: (9000, 28, 224, 1)
Test: (9000, 28, 224, 1)


## 4. Train model

In [11]:
units = 256

# Encoder (CNN)
img_input = Input(shape=(IMG_H, IMG_W, 1))          # (28, 224, 1)
x = Conv2D(32, (3, 3), padding='same', activation='relu')(img_input)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (14, 112, 32)
x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (7, 56, 64)
x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (3, 28, 128)

# Convert CNN features into a sequence
# feature map (H', W', C) -> chuỗi W' vector, mỗi vector mô tả một "cột" của ảnh
h, w, c = x.shape[1], x.shape[2], x.shape[3]
x = Permute((2, 1, 3))(x)
x = Reshape((w, h * c))(x)                          # (28, 3*128) = (28, 384)
enc_out = Dense(units, activation='relu')(x)          # (batch, W', units)      # (28, 256)
enc_out = Dropout(0.2)(enc_out)

# trạng thái khởi tạo cho decoder lấy từ trung bình các cột
enc_mean = GlobalAveragePooling1D()(enc_out)
init_h = Dense(units, activation='tanh')(enc_mean)
init_c = Dense(units, activation='tanh')(enc_mean)


# Decoder (LSTM + Attention)
dec_input = Input(shape=(max_len,))
emb = Embedding(N_CLASSES + 1, 64)(dec_input)
dec_out = LSTM(units, return_sequences=True)(emb, initial_state=[init_h, init_c])

# Attention: query = decoder, value = encoder
context = Attention()([dec_out, enc_out])
merged = Concatenate()([dec_out, context])
z = Dense(256, activation='relu')(merged)
outputs = Dense(N_CLASSES, activation='softmax')(z)

In [12]:
model = Model(inputs=[img_input, dec_input], outputs=outputs)
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

model.fit([X_train, D_train], y_train,
          epochs=50, 
          batch_size=64, 
          verbose=1,
          validation_data=([X_val, D_val], y_val),
          callbacks=[early_stop, reduce_lr])

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 28, 224,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 28, 224,   │        320 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 28, 224,   │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 14, 112,   │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 14, 112,   │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 14, 112,   │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 7, 56, 64) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 7, 56,     │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 7, 56,     │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 3, 28,     │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute (Permute)   │ (None, 28, 3,     │          0 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 28, 384)   │          0 │ permute[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 28, 256)   │     98,560 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 28, 256)   │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ dropout[0][0]     │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 8, 64)     │        768 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     65,792 │ global_average_p

 Total params: 787,339 (3.00 MB)

 Trainable params: 786,891 (3.00 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 61s 90ms/step - accuracy: 0.5871 - loss: 1.1481 - val_accuracy: 0.6924 - val_loss: 0.8210 - learning_rate: 0.0010
Epoch 2/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 75s 114ms/step - accuracy: 0.7602 - loss: 0.6135 - val_accuracy: 0.8016 - val_loss: 0.4947 - learning_rate: 0.0010
Epoch 3/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 78s 119ms/step - accuracy: 0.8044 - loss: 0.4928 - val_accuracy: 0.8256 - val_loss: 0.4399 - learning_rate: 0.0010
Epoch 4/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 74s 112ms/step - accuracy: 0.8257 - loss: 0.4366 - val_accuracy: 0.8109 - val_loss: 0.5067 - learning_rate: 0.0010
Epoch 5/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 69s 104ms/step - accuracy: 0.8399 - loss: 0.3993 - val_accuracy: 0.8480 - val_loss: 0.3820 - learning_rate: 0.0010
Epoch 6/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 63s 96ms/step - accuracy: 0.8496 - loss: 0.3733 - val_accuracy: 0.8557 - val_loss: 0.3674 - learning_rate: 0.0010
Epoch 7/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 74s 113ms/step - accuracy: 0.856

KeyboardInterrupt: 

In [1]:
import os
# Dòng này ra lệnh cho TensorFlow: "Hãy nói ra mọi suy nghĩ của mày, đừng giấu!"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0' 

import sys
import tensorflow as tf

print("👉 1. Đường dẫn Python Jupyter đang dùng:")
print(sys.executable)

print("\n👉 2. Phiên bản TensorFlow:")
print(tf.__version__)

print("\n👉 3. Đang dò tìm GPU (hãy nhìn kỹ dòng chữ bên dưới)...")
gpus = tf.config.list_physical_devices('GPU')
print("👉 4. Kết quả tìm thấy:", gpus)

I0000 00:00:1795194694.718514   83841 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795194694.743053   83841 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795194695.328053   83841 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


👉 1. Đường dẫn Python Jupyter đang dùng:
/home/pc/AAA/project/deepnn/forecast-practice/venv/bin/python

👉 2. Phiên bản TensorFlow:
2.21.0

👉 3. Đang dò tìm GPU (hãy nhìn kỹ dòng chữ bên dưới)...
👉 4. Kết quả tìm thấy: []


W0000 00:00:1795194696.109186   83841 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [2]:
import sys
print("Đường dẫn Python đang dùng:", sys.executable)

Đường dẫn Python đang dùng: /home/pc/AAA/project/deepnn/forecast-practice/venv/bin/python


## 5. Prediction

In [ ]:
# khi dự đoán không có nhãn thật nên phải sinh từng chữ số một,
# chữ số vừa sinh ra được đưa vào làm input cho bước kế tiếp
d = np.full((len(X_test), max_len), PAD_TOKEN, dtype='int32')
d[:, 0] = START_TOKEN
y_pred= np.zeros((len(X_test), max_len), dtype='int32')
for t in range(max_len):
    probs = model.predict([X_test, d], batch_size=256, verbose=0)
    y_pred[:, t] = np.argmax(probs[:, t, :], axis=-1)
    if t + 1 < max_len:
        d[:, t + 1] = y_pred[:, t]

## 6. Evaluation

In [ ]:
# Remove padding positions
mask = y_test != PAD_TOKEN
true_digits = y_test[mask].astype(np.float32)
pred_digits = np.where(y_pred[mask] == PAD_TOKEN, 0, y_pred[mask]).astype(np.float32)

In [ ]:
# Calculate evaluation metrics
# def report(name, y_true, y_hat):

# report("Individual digits", true_digits, pred_digits)
mse = mean_squared_error(true_digits, pred_digits)
nse = 1 - np.sum((true_digits - pred_digits) ** 2) / np.sum((true_digits - np.mean(true_digits)) ** 2)

print(f"R2: {r2_score(true_digits, pred_digits):.4f}")
print(f"NSE: {nse:.4f}")
print(f"MAE: {mean_absolute_error(true_digits, pred_digits):.4f}")
print(f"RMSE: {sqrt(mse):.4f}")

In [ ]:
# Convert digit sequences to integers
def to_number(row):
    digits = ''.join(str(d) for d in row if d != PAD_TOKEN)
    return int(digits) if digits else 0

true_num = np.array([to_number(row) for row in y_test], dtype=np.float64)
pred_num = np.array([to_number(row) for row in y_pred], dtype=np.float64)

mse = mean_squared_error(true_num, pred_num)
nse = 1 - np.sum((true_num - pred_num) ** 2) / np.sum((true_num - np.mean(true_num)) ** 2)

print(f"R2: {r2_score(true_num, pred_num):.4f}")
print(f"NSE: {nse:.4f}")
print(f"MAE: {mean_absolute_error(true_num, pred_num):.4f}")
print(f"RMSE: {sqrt(mse):.4f}")

In [ ]:
# Calculate accuracy
seq_acc = np.mean(np.all(y_pred == y_test, axis=1))
digit_acc = np.mean(y_pred[mask] == y_test[mask])

print(f"Sequence Accuracy: {seq_acc:.4f}")
print(f"Digit Accuracy: {digit_acc:.4f}")